# 01. Predict Model - Locally

Helper notebook for training fire risk model locally.

Recommended for making sure model + data are correct. Training full model locally is time consuming

IMPORTANT: Currently, multi-dir training shuffles features and labels so that they are mismatched
Need to fix either in geebeam export, in an offline post-processing step, or at training time by joining
based on index.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import tensorflow as tf
import keras
import aic_risk_modeling as arm
import matplotlib.pyplot as plt
import rasterio as rio
from rasterio.transform import Affine


In [ ]:
import json
with open('../out/fusion_test_v5.json', 'r') as f:
    config = json.load(f)

In [ ]:
SEED = 54
RNG = np.random.default_rng(SEED)

# Set params

In [ ]:
DATA_DIR = '../data/fusion_grid/allpreds_2025/'
OUTPUT_DIR = '../../test_out/2025/'
EDGE_CROP=8 # Crop each side by this (e.g. 0 if no crop)
CENTERED=True # True if x, y are for center for each tile
TFRECORD_PATTERN='*.tfrecord.gz'

In [ ]:
# Merged dataset test, merging along features-axis
training_ds = arm.train.build_merged_dataset([DATA_DIR], TFRECORD_PATTERN,  batch_size=4,  cache=False, axis='examples', shuffle=False)

In [ ]:
# Normalization
normalize_list = arm.train.get_normalize_list(config)
norm_func = arm.train.create_normalizer(DATA_DIR + '/stats.pbtxt', normalize_list)
training_ds = training_ds.map(norm_func)

In [ ]:
# Select bands
training_ds = arm.train.select_bands_transform(
    training_ds,
    input_feature_config=config['input_features'],
    output_feature_config=config['output_features']
)

In [ ]:
for inputs, labels in training_ds.take(2):
    print(inputs.keys())
    print(inputs.pop('md_sidecar', None))
    print(inputs.keys())

In [ ]:
new_model = keras.saving.load_model('../out/fusion_test_v5.keras', compile=False)

In [ ]:
all_outs = []
all_x = []
all_y = []
all_masks = []
for batch in training_ds:
    md_sidecar = batch[0].pop('md_sidecar')
    all_masks.append(np.array(batch[1]))
    all_x.append(np.array(md_sidecar[:, 0, 0]))
    all_y.append(np.array(md_sidecar[:, 0, 1]))
    out = new_model.predict_on_batch(batch[0])
    all_outs.append(out)

In [ ]:
print(len(all_outs))

In [ ]:
src = rio.open('../out/example.tif')

In [ ]:
profile = src.profile
profile.update(
    dtype=rio.float32,
    count=1,
    compress='lzw')
base_transform = profile['transform']

In [ ]:
# dtype to uint8, and specify LZW compression.
def write_batch(outs, masks, xs, ys):
    for i in range(len(outs)):
        out =outs[i]
        mask =masks[i]
        x = xs[i]
        y = ys[i]
        transform = Affine(base_transform[0], base_transform[1], x,
                                       base_transform[3], base_transform[4], y)
        if CENTERED:
            transform = transform*rio.Affine.translation(int(-out.shape[0]/2), int(-out.shape[1]/2))
        if EDGE_CROP > 0:
            out =out[EDGE_CROP:-EDGE_CROP, EDGE_CROP:-EDGE_CROP]
            mask =mask[EDGE_CROP:-EDGE_CROP, EDGE_CROP:-EDGE_CROP]
            transform = transform*rio.Affine.translation(8, 8)
        profile.update(dtype=rio.int8,
                       height=out.shape[0],
                       width=out.shape[1],
                       transform=transform)
        with rio.open(
            f'{OUTPUT_DIR}/mask_{x}-{y}.tif', 'w', **profile) as dst_dataset:
                dst_dataset.write(mask.astype(rio.int8), 1)

        profile.update(dtype=rio.float32)
        with rio.open(
            f'{OUTPUT_DIR}/out_{x}-{y}.tif', 'w', **profile) as dst_dataset:
                dst_dataset.write(out[:,:,0].astype(rio.float32), 1)


In [ ]:
for i in range(len(all_outs)):
    write_batch(all_outs[i], all_masks[i], all_x[i], all_y[i])